# NB02B — Primary benchmark: 3-fold within-site spatial cross-validation

Train **six independent ImageNet-pretrained ResNet18 U-Nets**: 3 folds × 2 supervision strategies (M1 partial-only and M2 partial + exhaustive training-fold annotations). Each fold holds out exhaustive circles in **every** site; their polygons and surrounding 15 m buffer are excluded from training RGB crops. MOS 3–4 excluded. Evaluate each of the 30 circles exactly once, using its held-out-fold model. Compare M1/M2 with paired canopy fraction, pixel ROC-AUC/AP, IoU, precision, recall and F1 at p≥0.5.

**Previous NB02A** is retained unchanged as a distinct site-transfer LOSO reference, not the primary accuracy estimate. No hyperparameter search on held-out circles.


## 01 — Inputs and NB01 spatial folds


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn segmentation-models-pytorch
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
OUT=ROOT/'Outputs/V3/02B_spatial3fold_no_MOS34';OUT.mkdir(parents=True,exist_ok=True)
REF=ROOT/'Outputs/V3/01_reference_data_no_MOS34'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
assert A.get('schema_version')=='v3_input_audit_5_nadine_circles'
assert A['exhaustive_annotations'].get('all_annotation_polygons_musanga') is True
assert int(A['exhaustive_annotations']['verified_exhaustive_circle_count'])==34
EXCLUDED_MOSAIC='Mos_3_4_transparent_mosaic_group1_georef.tif'
mosaics={m['file']:m for m in A['mosaics'] if m['file']!=EXCLUDED_MOSAIC}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
assert links.circle_id.nunique()==30
assert links.groupby('site').circle_id.nunique().to_dict()=={'Baego':7,'MOS':11,'Yangambi':7,'Yoko':5}
assert links.circle_id.is_unique
assert len(mosaics)==5,[(k,v['site']) for k,v in mosaics.items()]
TEST_SITE='SPATIAL_CV'  # training includes all four sites
FOLD=0;SEED=20261016;CROP=256;EPOCHS=25;BATCH=8
MAX_PARTIAL_PER_MOSAIC=250
MAX_EXHAUSTIVE_PER_SITE=260
HOLDOUT_COUNTS={'Baego':1,'MOS':2,'Yangambi':2,'Yoko':2}
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('PRIMARY: 3-fold within-site CV, partial-only M1 vs added-exhaustive M2')
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)




print('Mosaics:',[(name,m['site']) for name,m in mosaics.items()])


## 02 — Load annotations and ImageNet-pretrained ResNet18 U-Net

In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>(.01 if Path(path) in (Path(A['exhaustive_annotations']['complete_roi_boundaries']),Path(A['exhaustive_annotations']['positive_polygons']),Path(A['roi'])) else .05) or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
assert len(circles)==34 and len(positive)>0, (
    f'Unexpected reference geometries: circles={len(circles)} (expected 34), '
    f'Musanga polygons={len(positive)}; do not train until reconciled')
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))



In [ ]:
class ResNet18UNet(nn.Module):
    """Original main-branch U-Net: ImageNet-pretrained ResNet18 encoder, RGB in, 1 logit out."""
    def __init__(self):
        super().__init__()
        import segmentation_models_pytorch as smp
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',
                          in_channels=3,classes=1)
    def forward(self,x):
        return self.net(x)[:,0]

class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('ResNet18 U-Net ready (native 256×256 RGB crops).')




## 03 — Per-fold spatial exclusion (reconfigured for each training fold)

Use NB01 circle assignments, not site holdouts. Training crop footprint must not intersect any validation circle plus 15 m, across both MOS mosaics.


In [ ]:
assert links.groupby(['site','fold']).circle_id.nunique().ge(1).all()
assert sorted(links.fold.unique().astype(int).tolist())==[0,1,2]
HOLDOUT=set();DEV=set();BLOCKED={}
split_df=links.iloc[:0].copy()
print('NB01 circle folds:',links.groupby(['site','fold']).size().unstack(fill_value=0).to_dict())


## 04 — Resumable caches and direct RGB window reads

Source data, 256-pixel crops, augmentation, model architecture, optimizer and 25-epoch training remain unchanged. Cache fingerprint incorporates fold, supervision type and training-label selection. Direct Drive window reads avoid full-mosaic transfers.


In [ ]:
from rasterio.windows import Window
from torch.utils.data import Dataset,DataLoader
candidate=pd.read_csv(REF/'01_candidate_windows.csv')
candidate=candidate[candidate.mosaic.isin(mosaics)].copy()
all_circle_buffers={}
for name,d in geo.items():
    cs=circles.to_crs(d['crs'])
    all_circle_buffers[name]=[g.buffer(15.) for g in cs.geometry]
    d['dev_circles']=cs[cs.original_row.isin(DEV)]
    d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
    d['positive_local']=positive.to_crs(d['crs'])
    d['trees_local']=tree[d['site']].to_crs(d['crs'])
    d['tree_positive']=d['trees_local'].loc[d['trees_local'].class01==1]
    d['tree_negative']=d['trees_local'].loc[d['trees_local'].class01==0]
    d['roi_local']=roi.to_crs(d['crs'])

def burn_window(geometries,name,win):
    """Rasterize geometries intersecting a crop, using the spatial index.

    Accept GeoSeries OR GeoDataFrame; always iterate actual Shapely geometries,
    never GeoDataFrame column names.
    """
    d=geo[name]
    series=geometries.geometry if isinstance(geometries,gpd.GeoDataFrame) else geometries
    assert isinstance(series,gpd.GeoSeries),type(series)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    ids=series.sindex.query(bb,predicate='intersects')
    shape=(int(win.height),int(win.width))
    if len(ids)==0:return np.zeros(shape,dtype=bool)
    selected=series.iloc[ids]
    return rasterize(((g,1) for g in selected.values
                      if g is not None and not g.is_empty),
        out_shape=shape,transform=rasterio.windows.transform(win,d['transform']),
        fill=0,dtype='uint8').astype(bool)

_ACTIVE_RASTER=None
_ACTIVE_NAME=None

def rgb_for_window(name,win):
    if _ACTIVE_RASTER is not None and _ACTIVE_NAME==name:
        raw=_ACTIVE_RASTER.read([1,2,3],window=win,masked=True)
    else:
        with rasterio.open(geo[name]['path']) as ds:
            raw=ds.read([1,2,3],window=win,masked=True)
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    a=raw.filled(0).astype('float32')
    valid &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    valid &= ~((a[0]==0)&(a[1]==0)&(a[2]==255))
    return np.clip(a/255.,0,1).transpose(1,2,0).astype('float32'),valid

def training_crop(rec,model):
    name=rec.mosaic;d=geo[name]
    if rec.height<CROP or rec.width<CROP:return None
    rr=int(rec.row)+(int(rec.height)-CROP)//2
    cc=int(rec.col)+(int(rec.width)-CROP)//2
    win=Window(cc,rr,CROP,CROP)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    if any(bb.intersects(g) for g in BLOCKED[name]):return None
    if model=='M1' and any(bb.intersects(g) for g in all_circle_buffers[name]):return None
    if model=='M2' and rec.kind=='exhaustive' and int(rec.circle_id) not in DEV:return None
    inside=burn_window(d['roi_local'].geometry,name,win)
    pos_tree=burn_window(d['tree_positive'],name,win)
    neg_tree=burn_window(d['tree_negative'],name,win)
    if model=='M1':
        full=np.zeros_like(inside);pos_ex=np.zeros_like(inside)
    else:
        full=burn_window(d['dev_circles'].geometry,name,win)&inside
        pos_ex=burn_window(d['positive_local'].geometry,name,win)&full
    conflict=(pos_tree&neg_tree)| (pos_ex&neg_tree) | (full&~pos_ex&pos_tree)
    target=(pos_ex|(~full&pos_tree)).astype('float32')
    weight=(full|((pos_tree|neg_tree)&~full))&inside&~conflict
    img,valid=rgb_for_window(name,win)
    weight &= valid
    if int(weight.sum())<250:return None
    return dict(image=img,target=target,weight=weight.astype('float32'),
                site=d['site'],kind=rec.kind,mosaic=name)

def _pack_chips(path,items,signature):
    """Atomic, compressed uint8 cache of already-prepared chips."""
    if not items:return
    tmp=path.with_name(path.name+'.tmp.npz')
    try:
        np.savez_compressed(tmp,key=np.array(signature),
            rgb=np.stack([np.rint(np.clip(r['image'],0,1)*255).astype('uint8') for r in items]),
            target=np.stack([r['target'].astype('uint8') for r in items]),
            weight=np.stack([r['weight'].astype('uint8') for r in items]),
            site=np.array([r['site'] for r in items],dtype='U12'),
            kind=np.array([r['kind'] for r in items],dtype='U16'))
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)

def _unpack_chips(path,signature):
    with np.load(path,allow_pickle=False) as z:
        assert str(z['key'].item())==signature
        rgb=z['rgb'];target=z['target'];weight=z['weight']
        sites=z['site'].astype(str);kinds=z['kind'].astype(str)
        assert rgb.ndim==4 and rgb.shape[1:]==(CROP,CROP,3)
        assert target.shape==weight.shape==rgb.shape[:3]
        return [dict(image=rgb[i].astype('float32')/255.,
                     target=target[i].astype('float32'),
                     weight=weight[i].astype('float32'),
                     site=str(sites[i]),kind=str(kinds[i]))
                for i in range(len(rgb))]

import shutil
from contextlib import contextmanager
LOCAL_STAGE=Path('/content/musanga_02f_stage')
LOCAL_STAGE.mkdir(parents=True,exist_ok=True)
MIN_FREE_AFTER_STAGE=2*1024**3
USE_LOCAL_STAGE=False

@contextmanager
def staged_mosaic(name):
    """Stage one raster on SSD if it fits; use one open raster handle per mosaic."""
    global _ACTIVE_RASTER,_ACTIVE_NAME
    source=Path(geo[name]['path'])
    assert source.exists(),source
    free=shutil.disk_usage('/content').free
    needed=source.stat().st_size
    local=LOCAL_STAGE/(name.replace('/','_'))
    can_stage=USE_LOCAL_STAGE and free>=needed+MIN_FREE_AFTER_STAGE
    selected_path=source
    if can_stage:
        print(f'STAGE {name}: {needed/1e9:.2f} GB from Drive to local SSD '
              f'(available {free/1e9:.1f} GB)',flush=True)
        t0=time.perf_counter()
        tmp=local.with_name(local.name+'.part')
        try:
            with source.open('rb') as inp,tmp.open('wb') as out:
                shutil.copyfileobj(inp,out,length=16*1024*1024)
            assert tmp.stat().st_size==needed
            tmp.replace(local)
            selected_path=local
            print(f'STAGED {name} in {time.perf_counter()-t0:.1f}s',flush=True)
        except Exception as exc:
            tmp.unlink(missing_ok=True)
            local.unlink(missing_ok=True)
            print(f'Local copy failed; using original Drive image: {exc}',flush=True)
    else:
        print(f'STAGE SKIPPED {name}: image {needed/1e9:.2f} GB, '
              f'free SSD {free/1e9:.2f} GB; using one persistent Drive handle',
              flush=True)
    assert _ACTIVE_RASTER is None,'Another mosaic is already open.'
    try:
        with rasterio.open(selected_path) as ds:
            assert ds.count>=3 and ds.transform==geo[name]['transform']
            assert ds.crs==geo[name]['crs']
            _ACTIVE_RASTER=ds
            _ACTIVE_NAME=name
            yield
    finally:
        _ACTIVE_RASTER=None
        _ACTIVE_NAME=None
        if selected_path==local:
            local.unlink(missing_ok=True)
            print(f'RELEASED local SSD for {name}',flush=True)

class CropDataset(Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        q=self.rows[i];x=q['image'].copy();y=q['target'].copy();w=q['weight'].copy()
        k=int(torch.randint(0,4,(1,)).item())
        x=np.rot90(x,k);y=np.rot90(y,k);w=np.rot90(w,k)
        if torch.rand(1).item()<.5:x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))),
                torch.from_numpy(np.ascontiguousarray(y)),torch.from_numpy(np.ascontiguousarray(w)))



# Training subsets are drawn only from the NB01 fold-specific manifest.
import hashlib

# CONTENT signatures, not just paths/mtime: modified shapefile geometry or DBF
# triggers a new M2 training key and *both* M1/M2 evaluation keys.
def shapefile_content_signature(source):
    p=Path(source)
    if not p.is_file():raise FileNotFoundError(p)
    files=sorted(p.parent.glob(p.stem+'.*')) if p.suffix.lower()=='.shp' else [p]
    files=([x for x in files if x.suffix.lower() in ('.shp','.shx','.dbf','.prj','.cpg')]
           if p.suffix.lower()=='.shp' else [p])
    h=hashlib.sha256()
    for file in files:
        h.update(file.suffix.lower().encode())
        with file.open('rb') as stream:
            for chunk in iter(lambda:stream.read(4*1024*1024),b''):
                h.update(chunk)
    return h.hexdigest()

EXHAUSTIVE_SOURCE=Path(A['exhaustive_annotations']['positive_polygons'])
EXHAUSTIVE_SHA=shapefile_content_signature(EXHAUSTIVE_SOURCE)
print('Exhaustive annotation SHA256:',EXHAUSTIVE_SHA[:20],
      '| polygons loaded:',len(positive),flush=True)

CACHE_DIR=OUT/'chip_cache'
CACHE_DIR.mkdir(parents=True,exist_ok=True)
def cv_fingerprint(fold,model_type):
    sources=[REF/'01_cv_window_manifest.csv',REF/'01_circle_spatial_folds.csv',
             REF/'01_candidate_windows.csv',ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json']
    for name in sorted(mosaics):
        sources.append(Path(mosaics[name]['path']))
    deps=[(str(p),p.stat().st_size,p.stat().st_mtime_ns) for p in sources]
    payload=dict(version='spatial3fold_v2',fold=int(fold),model=model_type,
                 holdout=sorted(HOLDOUT),dev=sorted(DEV),crop=CROP,seed=SEED,
                 max_partial=MAX_PARTIAL_PER_MOSAIC,
                 max_exhaustive=MAX_EXHAUSTIVE_PER_SITE,buffer_m=15,files=deps,
                 exhaustive_annotations_sha256=EXHAUSTIVE_SHA if model_type=='M2' else None)
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def make_cv_cache(model_type,fold):
    key=cv_fingerprint(fold,model_type)
    cache_file=CACHE_DIR/f'{model_type}_fold{fold}_{key[:20]}.npz'
    if cache_file.exists():
        try:
            old=_unpack_chips(cache_file,key)
            assert len(old)>15
            assert all(x['kind']=='partial' for x in old) if model_type=='M1' else True
            print('CACHE HIT',model_type,'fold',fold,len(old),'chips',flush=True)
            return old
        except Exception as e:
            print('Invalid full cache, rebuilding:',repr(e),flush=True)
            cache_file.unlink(missing_ok=True)
    # Same NB01 train candidate windows for both models, with exhaustive windows
    # added exclusively in M2; validation-fold windows are never eligible.
    eligible=splits[(splits.cv_fold==fold)&(splits.split=='train')].copy()
    eligible=eligible[eligible.kind.isin(['partial'] if model_type=='M1' else ['partial','exhaustive'])]
    if model_type=='M2':
        eligible=eligible[(eligible.kind=='partial')|eligible.circle_id.isin(DEV)]
    chosen=[]
    for (name,kind),group in eligible.groupby(['mosaic','kind']):
        cap=MAX_PARTIAL_PER_MOSAIC*(3 if geo[name]['site']=='MOS' else 1) if kind=='partial' else MAX_EXHAUSTIVE_PER_SITE
        group=group.sample(n=min(cap,len(group)),random_state=SEED)
        chosen.append(group)
    assert chosen,'No CV training windows'
    selected=pd.concat(chosen,ignore_index=True)
    assert not selected[selected.kind=='exhaustive'].circle_id.isin(HOLDOUT).any()
    print(model_type,'fold',fold,'selected windows:',selected.groupby(['site','kind']).size().to_dict(),flush=True)
    parts_dir=CACHE_DIR/f'{model_type}_fold{fold}_{key[:20]}'
    parts_dir.mkdir(parents=True,exist_ok=True)
    all_chips=[]
    for name,grp in selected.groupby('mosaic',sort=True):
        records=list(grp.itertuples(index=False))
        with staged_mosaic(name):
            for off in range(0,len(records),100):
                part=parts_dir/f'{name}_{off:04d}.npz'
                if part.exists():
                    try:
                        chunks=_unpack_chips(part,key)
                        all_chips.extend(chunks)
                        continue
                    except Exception:
                        part.unlink(missing_ok=True)
                items=[x for row in records[off:off+100] if (x:=training_crop(row,model_type)) is not None]
                if items:_pack_chips(part,items,key)
                all_chips.extend(items)
                print(model_type,'fold',fold,name,min(off+100,len(records)),'/',len(records),'usable',len(items),flush=True)
    assert len(all_chips)>15,(model_type,fold,len(all_chips))
    assert all(chip['kind']=='partial' for chip in all_chips) if model_type=='M1' else True
    _pack_chips(cache_file,all_chips,key)
    print('CACHE SAVED',model_type,'fold',fold,len(all_chips),'chips',flush=True)
    return all_chips


## 05 — Identical training recipe for M1 and M2; three spatial folds


In [ ]:
import time
from contextlib import nullcontext
USE_AMP=(device=='cuda')
# float16 is fast on both T4 and L4; BCE is evaluated in fp32 for stability.
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
print('CUDA mixed precision:',USE_AMP,'| GPU:',torch.cuda.get_device_name(0),flush=True)


def fit_cv(model_type,fold):
    fold_out=BASE_OUT/f'fold_{fold}'
    fold_out.mkdir(parents=True,exist_ok=True)
    key=cv_fingerprint(fold,model_type)
    # Versioned M2 models cannot silently reuse old Yangambi labels.
    # M1 uses partial labels only. Reuse the original completed M1 checkpoint
    # under the explicit assumption that only exhaustive POSITIVE polygons changed.
    legacy_m1=fold_out/f'fold_{fold}'/'M1_resnet18_spatialCV.pt'
    ckpt=(legacy_m1 if model_type=='M1' and legacy_m1.exists() else
          fold_out/f'M1_resnet18_spatialCV.pt' if model_type=='M1' else
          fold_out/f'M2_resnet18_spatialCV_{key[:20]}.pt')
    torch.manual_seed(SEED)
    model=ResNet18UNet().to(device)
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt,map_location=device,weights_only=True))
        print('Reusing spatial-CV checkpoint:',ckpt,flush=True)
        if model_type=='M1':
            print('M1 reuse assumes partial tree annotations, circle boundaries, '
                  'fold allocation and ROI unchanged.',flush=True)
        return model.eval()
    chips=make_cv_cache(model_type,fold)
    loader=DataLoader(CropDataset(chips),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
    optim=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
    local_scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
    history=[]
    for ep in range(EPOCHS):
        model.train();losses=[];started=time.perf_counter()
        for x,y,w in loader:
            x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True);w=w.to(device,non_blocking=True)
            optim.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=USE_AMP):
                logits=model(x)
            loss=masked_loss(logits.float(),y,w)
            local_scaler.scale(loss).backward()
            local_scaler.step(optim);local_scaler.update()
            losses.append(float(loss.detach()))
        avg=float(np.mean(losses))
        history.append(dict(epoch=ep+1,masked_BCE=avg,seconds=time.perf_counter()-started))
        pd.DataFrame(history).to_csv(fold_out/f'{model_type}_loss.csv',index=False)
        print(f'{model_type} fold {fold} epoch {ep+1:02d}/{EPOCHS} BCE={avg:.4f} {history[-1]["seconds"]:.1f}s',flush=True)
    torch.save(model.state_dict(),ckpt)
    del chips
    print('Saved:',ckpt,flush=True)
    return model.eval()


## 06 — Independent circle inference: each circle evaluated exactly once per model


In [ ]:
from scipy.special import expit,logit
from scipy.optimize import minimize
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
WINDOWS=pd.read_csv(REF/'01_candidate_windows.csv')
VAL_WINDOWS=WINDOWS[WINDOWS.kind=='exhaustive'].copy()
VAL_WINDOWS=VAL_WINDOWS[VAL_WINDOWS.circle_id.isin(links.circle_id)].copy()
assert set(VAL_WINDOWS.circle_id.unique().astype(int))==set(links.circle_id.astype(int))
N_BINS=256
BINS=np.linspace(0,1,N_BINS+1)
BIN_CENTER=(BINS[:-1]+BINS[1:])/2

def run_all_circles(model,model_name):
    statfile=OUT/f'{model_name}_spatialCV_circle_raw_{EVAL_KEY[:20]}.csv'
    histfile=OUT/f'{model_name}_spatialCV_histograms_{EVAL_KEY[:20]}.npz'
    if statfile.exists() and histfile.exists():
        return pd.read_csv(statfile),dict(np.load(histfile))
    model.eval();rows=[];histograms={}
    for _,circle in split_df.iterrows():
        cid=int(circle.circle_id)
        name=circle.mosaic
        d=geo[name]
        windows=VAL_WINDOWS[(VAL_WINDOWS.circle_id==cid)&(VAL_WINDOWS.mosaic==name)]
        assert len(windows),f'No evaluation windows for circle {cid}'
        cg=circles[circles.original_row==cid].to_crs(d['crs'])
        counts=np.zeros(N_BINS,np.int64);true_counts=np.zeros(N_BINS,np.int64)
        n=0;sumy=0.;sump=0.;sum_sq=0.;tp=fp=fn=tn=0
        rng=np.random.default_rng(SEED+cid)
        sample_truth=[];sample_probs=[]
        for rec in windows.itertuples(index=False):
            win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
            mask=burn_window(cg.geometry,name,win)
            mask &= burn_window(d['roi_local'].geometry,name,win)
            rgb,valid=rgb_for_window(name,win)
            mask &= valid
            if not mask.any():continue
            reference=burn_window(d['positive_local'].geometry,name,win)
            x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
            with torch.inference_mode():
                pred=torch.sigmoid(model(x))[0].cpu().numpy()
            y=reference[mask].astype('uint8');p=pred[mask].astype('float32')
            n+=len(y);sumy+=float(y.sum());sump+=float(p.sum())
            sum_sq+=float(np.square(p-y).sum())
            binary=p>=.5
            tp+=int(np.sum(binary&(y==1)));fp+=int(np.sum(binary&(y==0)))
            fn+=int(np.sum((~binary)&(y==1)))
            tn+=int(np.sum((~binary)&(y==0)))
            ix=np.minimum((p*N_BINS).astype(np.int32),N_BINS-1)
            counts+=np.bincount(ix,minlength=N_BINS).astype('int64')
            true_counts+=np.bincount(ix,weights=y,minlength=N_BINS).astype('int64')
            # only diagnostic AUC/AP uses pixel subsampling, with fixed per-circle cap
            take=rng.choice(len(y),min(100,len(y)),replace=False)
            sample_truth.extend(y[take].tolist());sample_probs.extend(p[take].tolist())
        assert n>0,f'No valid ROI/RGB reference pixels for circle {cid}'
        assert int(counts.sum())==n
        yt=np.asarray(sample_truth);pr=np.asarray(sample_probs)
        rows.append(dict(model=model_name,circle_id=cid,site=circle.site,split=circle.split,
                         valid_pixels=n,observed_fraction=100*sumy/n,
                         predicted_fraction=100*sump/n,
                         bias_pp=100*(sump-sumy)/n,brier=sum_sq/n,
                         f1_threshold_05=2*tp/max(1,2*tp+fp+fn),
                         tp=tp,fp=fp,fn=fn,tn=tn,
                         iou_threshold_05=tp/max(1,tp+fp+fn),
                         precision_threshold_05=tp/max(1,tp+fp),
                         recall_threshold_05=tp/max(1,tp+fn),
                         auc_sample=roc_auc_score(yt,pr) if len(np.unique(yt))>1 else np.nan,
                         ap_sample=average_precision_score(yt,pr) if len(np.unique(yt))>1 else np.nan))
        histograms[str(cid)+'_count']=counts
        histograms[str(cid)+'_positive']=true_counts
        print(model_name,'circle',cid,'obs',round(100*sumy/n,1),
              'pred',round(100*sump/n,1),'valid pixels',n,flush=True)
    frame=pd.DataFrame(rows)
    assert frame.circle_id.is_unique
    assert set(frame.circle_id.astype(int))==set(split_df.circle_id.astype(int))
    assert len(frame)==len(split_df)
    frame.to_csv(statfile,index=False)
    np.savez_compressed(histfile,**histograms)
    return frame,histograms



In [ ]:
# PRIMARY spatial-CV experiment: site-stratified circle folds (not LOSO).
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import gc
SITES=['Baego','MOS','Yangambi','Yoko']
FOLDS=[0,1,2]
BASE_OUT=OUT
# Different exhaustive polygon bytes => different evaluation cache signature.
# Also account for circle boundaries / ROI / validation windows (if modified).
def evaluation_fingerprint(fold,model_type):
    validation_sources=[Path(A['exhaustive_annotations']['complete_roi_boundaries']),
                        Path(A['roi']),REF/'01_candidate_windows.csv',
                        REF/'01_circle_spatial_folds.csv']
    h=hashlib.sha256()
    h.update(json.dumps(dict(fold=int(fold),model=model_type,
              source_sha=EXHAUSTIVE_SHA,
              training_key=cv_fingerprint(fold,model_type)),sort_keys=True).encode())
    for p in validation_sources:
        h.update(shapefile_content_signature(p).encode())
    return h.hexdigest()

print('Reference annotation version:',EXHAUSTIVE_SHA[:20],flush=True)
assert sorted(links.fold.unique().astype(int))==FOLDS
assert links.groupby(['site','fold']).circle_id.nunique().ge(1).all()

ALL_SCORES=[]
ALL_HISTS={}
for cv_fold in FOLDS:
    FOLD=int(cv_fold)
    HOLDOUT=set(links.loc[links.fold==FOLD,'circle_id'].astype(int))
    DEV=set(links.loc[links.fold!=FOLD,'circle_id'].astype(int))
    assert HOLDOUT.isdisjoint(DEV) and len(HOLDOUT|DEV)==30
    split_df=links[links.fold==FOLD][['circle_id','site','mosaic','cover_pct']].copy()
    split_df['split']='held_out_spatial_fold'
    for name,d in geo.items():
        cs=circles.to_crs(d['crs'])
        d['dev_circles']=cs[cs.original_row.isin(DEV)]
        # Include all validation circles geometrically, not just own-mosaic circles.
        BLOCKED[name]=[g.buffer(15) for g in cs[cs.original_row.isin(HOLDOUT)].geometry]
    OUT=BASE_OUT/f'fold_{FOLD}'
    OUT.mkdir(parents=True,exist_ok=True)
    print('\n===== SPATIAL CV FOLD',FOLD,'holdout circles',len(HOLDOUT),'training circles',len(DEV),'=====',flush=True)
    for model_type in ['M1','M2']:
        EVAL_KEY=evaluation_fingerprint(FOLD,model_type)
        print('Evaluation signature',model_type,FOLD,EVAL_KEY[:20],flush=True)
        model=fit_cv(model_type,FOLD)
        result,h=run_all_circles(model,model_type)
        assert set(result.circle_id.astype(int))==HOLDOUT
        result['fold']=FOLD
        ALL_SCORES.append(result)
        for cid in HOLDOUT:
            for suffix in ['count','positive']:
                ALL_HISTS[f'{model_type}_{cid}_{suffix}']=h[f'{cid}_{suffix}']
        pd.concat(ALL_SCORES,ignore_index=True).to_csv(BASE_OUT/'02B_spatialCV_progress.csv',index=False)
        print('Completed',model_type,'fold',FOLD,'circles',len(result),flush=True)
        del model;torch.cuda.empty_cache();gc.collect()

SCORES=pd.concat(ALL_SCORES,ignore_index=True)
assert SCORES.groupby('model').circle_id.nunique().to_dict()=={'M1':30,'M2':30}
assert not SCORES.duplicated(['model','circle_id']).any()
SCORES.to_csv(BASE_OUT/f'02B_spatialCV_all30_paired_{EXHAUSTIVE_SHA[:12]}.csv',index=False)
SCORES.to_csv(BASE_OUT/'02B_spatialCV_all30_paired.csv',index=False)
np.savez_compressed(BASE_OUT/'02B_spatialCV_histograms.npz',**ALL_HISTS)

def pooled_hist_metric(m,ids):
    c=np.sum([ALL_HISTS[f'{m}_{int(cid)}_count'] for cid in ids],axis=0).astype('float64')
    p=np.sum([ALL_HISTS[f'{m}_{int(cid)}_positive'] for cid in ids],axis=0).astype('float64')
    n=c-p;np_=p.sum();nn=n.sum()
    if np_==0 or nn==0:return (np.nan,np.nan)
    auc=float(np.dot(p,np.cumsum(n)-0.5*n)/(np_*nn))
    cp=np.cumsum(p[::-1]);cn=np.cumsum(n[::-1])
    ap=float(np.sum(p[::-1]/np_*cp/np.maximum(cp+cn,1)))
    return auc,ap

rows=[]
for model_type in ['M1','M2']:
    for site in ['ALL']+SITES:
        g=SCORES[SCORES.model==model_type]
        if site!='ALL':g=g[g.site==site]
        tp,fp,fn=(int(g[k].sum()) for k in ['tp','fp','fn'])
        auc,ap=pooled_hist_metric(model_type,g.circle_id)
        rows.append(dict(model=model_type,site=site,circles=len(g),
            auc=auc,ap=ap,pixel_iou=tp/max(1,tp+fp+fn),
            pixel_f1=2*tp/max(1,2*tp+fp+fn),
            pixel_precision=tp/max(1,tp+fp),pixel_recall=tp/max(1,tp+fn),
            fraction_R2=r2_score(g.observed_fraction,g.predicted_fraction) if len(g)>=2 else np.nan,
            fraction_MAE_pp=mean_absolute_error(g.observed_fraction,g.predicted_fraction),
            fraction_RMSE_pp=np.sqrt(mean_squared_error(g.observed_fraction,g.predicted_fraction)),
            fraction_bias_pp=float(g.bias_pp.mean())))
SUMMARY=pd.DataFrame(rows)
display(SUMMARY.round(4))
SUMMARY.to_csv(BASE_OUT/f'02B_spatialCV_summary_by_site_{EXHAUSTIVE_SHA[:12]}.csv',index=False)
SUMMARY.to_csv(BASE_OUT/'02B_spatialCV_summary_by_site.csv',index=False)
paired=SCORES[SCORES.model=='M1'][['circle_id','site','fold','observed_fraction','predicted_fraction','bias_pp','iou_threshold_05','f1_threshold_05']].merge(
    SCORES[SCORES.model=='M2'][['circle_id','observed_fraction','predicted_fraction','bias_pp','iou_threshold_05','f1_threshold_05']],
    on='circle_id',suffixes=('_M1','_M2'),validate='one_to_one')
assert np.allclose(paired.observed_fraction_M1,paired.observed_fraction_M2,atol=0.02)
paired.to_csv(BASE_OUT/'02B_spatialCV_paired_circle_comparison.csv',index=False)
(BASE_OUT/'02B_latest_reference_version.json').write_text(json.dumps(
    dict(exhaustive_source=str(EXHAUSTIVE_SOURCE),sha256=EXHAUSTIVE_SHA,
         circles=30,models=['M1','M2'],folds=FOLDS),indent=2))
print('PRIMARY BENCHMARK COMPLETE: 3 within-site spatial folds × M1/M2, all 30 circles scored independently.',BASE_OUT)


## 07 — Publication figures and additional performance metrics (evaluation only)

Reads the saved **30 independent, out-of-fold circle predictions** and writes reproducible publication-quality PNG/PDF figures. Figures: site-coloured **M1/M2 observed–predicted** scatter (circle IDs; 1:1 line), paired circle error-change plot, faceted site scatters, and fold-level mean ± SD with individual fold points. Per-circle uncertainty bars **are not estimated**: each circle was withheld in exactly one CV fold and has one prediction per model.

Additional metrics: **RPD** = sample standard deviation of observed fractional cover / RMSE (dimensionless), **RPIQ** = observed-cover IQR / RMSE, fraction nRMSE relative to observed range, descriptive calibration slope/intercept, and per-site as well as pooled metrics. RPD and RPIQ describe prediction relative to variability of the validation observations; they can look poor in homogeneous old-growth stands despite moderate absolute errors. With just 5–11 circles/site, interpret site RPD and R² cautiously.

All visualization analyses are descriptive and do not tune classification thresholds or models on the held-out reference circles. RGB-specific cross-site comparisons remain in NB02A.

In [ ]:
# NB02B Block 07: publication figures + metric audit; NO model training
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from IPython.display import display

REPORT_DIR=ROOT/'Outputs/V3/02B_spatial3fold_no_MOS34'
FIG_DIR=REPORT_DIR/'figures'
FIG_DIR.mkdir(parents=True,exist_ok=True)
READ=REPORT_DIR/'02B_spatialCV_all30_paired.csv'
assert READ.exists(),f'Run NB02B CV first: {READ}'
d=pd.read_csv(READ)
required={'model','site','circle_id','fold','observed_fraction','predicted_fraction','tp','fp','fn'}
assert required.issubset(d.columns),required-set(d.columns)
assert len(d)==60 and d.groupby('model').circle_id.nunique().to_dict()=={'M1':30,'M2':30}
assert not d.duplicated(['model','circle_id']).any()
assert d.groupby(['model','fold']).circle_id.nunique().min()>=1
assert sorted(d.fold.unique())==[0,1,2]

site_order=['Baego','MOS','Yangambi','Yoko']
colors={'Baego':'#3168B4','MOS':'#E28E23','Yangambi':'#27856A','Yoko':'#AB477B'}
symbols={'M1':'o','M2':'^'}
models=['M1','M2']
plt.rcParams.update({'font.size':10,'axes.titlesize':12,'axes.labelsize':11,
                     'savefig.dpi':200,'pdf.fonttype':42})
def savefig(fig,name):
    for suffix in ('png','pdf'):
        fig.savefig(FIG_DIR/(name+'.'+suffix),dpi=220,bbox_inches='tight')
    plt.show()
    plt.close(fig)
def fraction_metrics(sub):
    yy=sub.observed_fraction.to_numpy(float)
    pp=sub.predicted_fraction.to_numpy(float)
    error=pp-yy
    rmse=float(np.sqrt(np.mean(error**2)))
    sd=float(np.std(yy,ddof=1)) if len(yy)>1 else np.nan
    iqr=float(np.percentile(yy,75)-np.percentile(yy,25))
    value_range=float(np.max(yy)-np.min(yy))
    slope,intercept=np.polyfit(yy,pp,1) if len(yy)>1 and sd>0 else (np.nan,np.nan)
    tp,fp,fn=(float(sub[k].sum()) for k in ('tp','fp','fn'))
    return dict(n=len(sub),R2=float(r2_score(yy,pp)) if len(yy)>1 and sd>0 else np.nan,
                MAE_pp=float(np.mean(abs(error))),RMSE_pp=rmse,bias_pp=float(np.mean(error)),
                observed_sd_pp=sd,observed_IQR_pp=iqr,
                RPD=sd/rmse if rmse>0 else np.nan,
                RPIQ=iqr/rmse if rmse>0 else np.nan,
                nRMSE_range=rmse/value_range if value_range>0 else np.nan,
                calibration_slope=float(slope),calibration_intercept_pp=float(intercept),
                pixel_F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan,
                pixel_IoU=tp/(tp+fp+fn) if tp+fp+fn else np.nan,
                pixel_precision=tp/(tp+fp) if tp+fp else np.nan,
                pixel_recall=tp/(tp+fn) if tp+fn else np.nan)
metric_rows=[]
for model in models:
    sub=d[d.model==model]
    metric_rows.append(dict(model=model,site='ALL',**fraction_metrics(sub)))
    for site in site_order:
        metric_rows.append(dict(model=model,site=site,
                                **fraction_metrics(sub[sub.site==site])))
METRICS=pd.DataFrame(metric_rows)
METRICS.to_csv(REPORT_DIR/'02B_publication_performance_summary.csv',index=False)
print('=== Fraction cover, spatial F1/IoU, RPD and RPIQ ===')
display(METRICS.round(3))

left=d[d.model=='M1'].set_index('circle_id')
right=d[d.model=='M2'].set_index('circle_id')
assert left.index.is_unique and right.index.is_unique and set(left.index)==set(right.index)
assert (left.loc[right.index,'site'].to_numpy()==right.site.to_numpy()).all()
assert (left.loc[right.index,'fold'].to_numpy()==right.fold.to_numpy()).all()
paired=pd.DataFrame({
    'circle_id':right.index,
    'site':right.site.values,'fold':right.fold.values,
    'observed':right.observed_fraction.values,
    'pred_M1':left.loc[right.index,'predicted_fraction'].values,
    'pred_M2':right.predicted_fraction.values})
assert np.allclose(paired.observed,left.loc[right.index,'observed_fraction'],atol=0.02)
paired['error_M1']=paired.pred_M1-paired.observed
paired['error_M2']=paired.pred_M2-paired.observed
paired['abs_error_M1']=abs(paired.error_M1)
paired['abs_error_M2']=abs(paired.error_M2)
paired['improvement_pp']=paired.abs_error_M1-paired.abs_error_M2  # positive = M2 better
paired.sort_values(['site','circle_id']).to_csv(
    REPORT_DIR/'02B_publication_paired_circle_errors.csv',index=False)
print('M2 lower absolute error in',int((paired.improvement_pp>0).sum()),'of 30 circles.')

# Figure 1: two-panel site-coloured scatter, annotated by circle number
fig,axes=plt.subplots(1,2,figsize=(13,6),sharex=True,sharey=True,layout='constrained')
for ax,model in zip(axes,models):
    sub=d[d.model==model]
    m=METRICS[(METRICS.model==model)&(METRICS.site=='ALL')].iloc[0]
    for site in site_order:
        ss=sub[sub.site==site]
        ax.scatter(ss.observed_fraction,ss.predicted_fraction,
                   s=65,color=colors[site],alpha=.87,edgecolors='white',linewidths=.5)
        for q in ss.itertuples(index=False):
            ax.annotate(str(int(q.circle_id)),(q.observed_fraction,q.predicted_fraction),
                        xytext=(4,3),textcoords='offset points',fontsize=8,color=colors[site])
    ax.plot([0,100],[0,100],ls='--',lw=1.2,color='0.35')
    ax.set(xlim=(-2,103),ylim=(-2,103),aspect='equal',xlabel='Observed Musanga cover (%)',
           ylabel='Predicted Musanga cover (%)',
           title=f'{model} · R²={m.R2:.3f}, MAE={m.MAE_pp:.1f} pp')
    ax.grid(alpha=.16)
legend=[Line2D([0],[0],marker='o',color='none',markerfacecolor=colors[s],
               markersize=8,label=s) for s in site_order]
fig.legend(handles=legend,loc='outside lower center',ncol=4,frameon=False)
fig.suptitle('Musanga cover: 30 spatially held-out exhaustive circles')
savefig(fig,'fig01_observed_predicted_M1_M2_sites')

# Figure 2: paired absolute error, site colours and circle IDs
fig,ax=plt.subplots(figsize=(7,6.5),layout='constrained')
hi=float(np.ceil(max(paired.abs_error_M1.max(),paired.abs_error_M2.max())+3))
for site in site_order:
    ss=paired[paired.site==site]
    ax.scatter(ss.abs_error_M1,ss.abs_error_M2,s=70,color=colors[site],alpha=.88,
               label=site,edgecolor='white',linewidth=.6)
    for q in ss.itertuples(index=False):
        ax.annotate(str(int(q.circle_id)),(q.abs_error_M1,q.abs_error_M2),
                    xytext=(4,3),textcoords='offset points',fontsize=8,color=colors[site])
ax.plot([0,hi],[0,hi],ls='--',color='0.35',lw=1.2)
ax.set(xlim=(0,hi),ylim=(0,hi),aspect='equal',
       xlabel='M1 absolute fraction error (percentage points)',
       ylabel='M2 absolute fraction error (percentage points)',
       title='Circle-level errors: below diagonal = M2 improves')
ax.legend(frameon=False);ax.grid(alpha=.15)
savefig(fig,'fig02_paired_absolute_errors')

# Figure 3: per-site faceted scatter for M2, with consistent 1:1 axes
fig,axes=plt.subplots(2,2,figsize=(10.4,9.4),sharex=True,sharey=True,layout='constrained')
for ax,site in zip(axes.flat,site_order):
    ss=d[(d.model=='M2')&(d.site==site)]
    ms=METRICS[(METRICS.model=='M2')&(METRICS.site==site)].iloc[0]
    ax.scatter(ss.observed_fraction,ss.predicted_fraction,color=colors[site],
               s=72,alpha=.9,edgecolor='white',linewidths=.5)
    for q in ss.itertuples(index=False):
        ax.annotate(str(int(q.circle_id)),(q.observed_fraction,q.predicted_fraction),
                    xytext=(5,4),textcoords='offset points',fontsize=9)
    ax.plot([0,100],[0,100],ls='--',color='0.45')
    ax.set(xlim=(-2,103),ylim=(-2,103),aspect='equal',
           title=f'{site} (n={len(ss)})\nMAE={ms.MAE_pp:.1f} pp · R²={ms.R2:.2f}')
    ax.grid(alpha=.17)
for ax in axes[-1,:]:ax.set_xlabel('Observed cover (%)')
for ax in axes[:,0]:ax.set_ylabel('Predicted cover (%)')
fig.suptitle('M2 within-site performance and circle IDs')
savefig(fig,'fig03_M2_site_facets')

# Figure 4: three independent fold metrics (points), fold mean ± descriptive SD
fold_rows=[]
for (model,fold),ss in d.groupby(['model','fold']):
    fold_rows.append(dict(model=model,fold=int(fold),**fraction_metrics(ss)))
FOLD_METRICS=pd.DataFrame(fold_rows).sort_values(['model','fold'])
FOLD_METRICS.to_csv(REPORT_DIR/'02B_publication_fold_metrics.csv',index=False)
print('=== Spatial fold metrics (descriptive variability; n=3 folds) ===')
display(FOLD_METRICS[['model','fold','n','R2','MAE_pp','RMSE_pp','pixel_F1','pixel_IoU','RPD']].round(3))
fig,axes=plt.subplots(1,3,figsize=(12.7,4),layout='constrained')
for ax,(metric,label) in zip(axes,[('MAE_pp','Fraction MAE (pp)'),
                                    ('pixel_F1','Pixel F1 at p=0.5'),
                                    ('R2','Fraction R²')]):
    for ix,model in enumerate(models):
        values=FOLD_METRICS[FOLD_METRICS.model==model][metric].to_numpy(float)
        ax.errorbar(ix,float(values.mean()),yerr=float(values.std(ddof=1)),
                    fmt='s',markersize=8,capsize=5,color=['#566A8C','#BC7035'][ix])
        ax.scatter(np.full(3,ix)+np.array([-.065,0,.065]),values,
                   marker='o',s=26,color=['#566A8C','#BC7035'][ix],alpha=.7)
    ax.set_xticks([0,1],models);ax.set_ylabel(label);ax.grid(axis='y',alpha=.17)
fig.suptitle('Spatial CV: individual folds and mean ± SD (3 folds, descriptive)')
savefig(fig,'fig04_fold_variability')

print('Saved metric CSVs and publication figures to:',FIG_DIR)
